Try to Implement FireGnn on HeteroGraph

In [1]:
import pickle
import sys
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, GATConv, GINConv
from torch_geometric.data import Data
from torch_geometric.nn import HeteroConv, GATConv, HGTConv
from torch_geometric.data import HeteroData
from torch_scatter import scatter_sum, scatter_mean, scatter_max
import numpy as np
import networkx as nx

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import KMeans
from collections import Counter
import scipy.stats
from tqdm import tqdm
import pandas as pd
import re
from typing import Any, Dict, Tuple

/opt/anaconda3/envs/firegnn/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm




### Design
+ Graph: Map patients as nodes onto AD-KG: edges are created between patient and proteins if the protein exist in gene-expression data;
    + patient_node.y = labels;
    + patient_protein_edge_weight = gene expression values;
    + all the other nodes: node.y = (association_score_to_AD > 0.5)?
    + 
+ HeteroData:
    + data.x:
    + data.y:
    + data.relevance:
    + data.topo_features:

#### (a) using conditional rules to give various emphasis on patient-protein edge weight
+ ConditioanlMessageLayer:
    + Conditional message passing:
    + patient - protein edge:
      + $ r_1(v) $ = relevance score of non-patient node;
      + $ r_2(u) = \begin{cases} 1 & \text{if }  y_u = 1 \\ 0 & \text{if }  y_u = 0 \end{cases}$, where $u$ is patient ndoe;
      + $ r_3(u,v) = $e_{u,v}, the edge_weight between node $u$ and $v$;
      + combined gating rule:
        + $ g_{u,v} = \sigma(\alpha (r_1(v)*r_3(u,v) - \theta)) * r_2(u) $; 
        + $ g_{u,v} \in [0,1]$
    + all other edges gating:
      + $g_{u,v}$ = edge weight
        + edge_weight = embedding similarity between nodes;(`current implementation`)
        + edge_weight = the average node_relevance of two nodes;(`prefer`)
        + or edge_weight = 1: `not good because gated_edge_weight of edge_patient&protein < 1`
  
+ GNNModel
  + instead of combining gate-layer and GNN layer before the final classifier layer, here the rule-layer should be incorporated in each GNN layer;
  + incorporate gated_edge_weight to GAT layer.

#### (b) Message Passing Gating to each GAT layer
+ ConditionalMessageLayer
+ simple_GATConv combine the output of ConditioanlMessageLayer
+ HeteroGATModel to wrap all simple_GATConv for all edge_types
#### (c) Conditional Attention coefficient
+ still need to explore

## Prepare Patient-Graph

load data

In [2]:
ad_kg_path = "./data/KG/ad_network_with_reverse_edges.pkl"
adni_exp_path = "./data/adni_gene_cleaned.csv"
adni_target_path = "./data/adni_targets.tsv"

# load graph
with open(ad_kg_path, 'rb') as f:
    kg = pickle.load(f)
# read patient gene expression data
exp = pd.read_csv(adni_exp_path, index_col=0)
exp = exp.transpose()

# read patient labels
design = pd.read_csv(adni_target_path, sep='\t', index_col = 0)
design['Target'] = design['Target'].map({'Control':0, 'Disease':1})
labels = design['Target'].to_list()

In [3]:
exp_log = np.log1p(exp)
exp_zscore = (exp_log - exp_log.mean())/exp_log.std()
exp_minmax = (exp_log - exp_log.min())/(exp_log.max()-exp_log.min())

exp_minmax

genes,A1BG,A1CF,A2M,A2ML1,A3GALT2,A4GALT,A4GNT,AAAS,AACS,AACSP1,...,ZW10,ZWILCH,ZWINT,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3
116_S_1249,0.571671,0.340615,0.175825,0.626066,0.542598,0.079507,0.523829,0.615859,0.545449,0.190199,...,0.911154,0.472502,0.233955,0.644354,0.648553,0.168286,0.382826,0.528092,0.811732,0.821232
037_S_4410,0.312765,0.107364,0.402568,0.435299,0.164096,0.199824,0.057381,0.475350,0.516715,0.285832,...,0.965697,0.681593,0.385317,0.757155,0.576793,0.188903,0.440990,0.129150,0.526449,0.918164
006_S_4153,0.367589,0.434225,0.396191,0.384513,0.368071,0.288051,0.208212,0.547245,0.457690,0.462182,...,0.749151,0.548002,0.448831,0.559800,0.592951,0.459771,0.634291,0.477557,0.829727,0.675254
116_S_1232,0.420091,0.458687,0.522685,0.354384,0.346321,0.335564,0.730456,0.699923,0.493436,0.476690,...,0.689930,0.454267,0.386036,0.431721,0.626827,0.305865,0.502166,0.574257,0.743878,0.503902
099_S_4205,0.412820,0.477623,0.337388,0.380757,0.430647,0.220729,0.278801,0.552444,0.409338,0.310235,...,0.857544,0.565459,0.413879,0.835155,0.678807,0.293313,0.649688,0.441321,0.761394,0.735266
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
009_S_2381,0.381246,0.638092,0.394487,0.500229,0.624854,0.553319,0.533921,0.492256,0.352875,0.545118,...,0.579482,0.321750,0.395003,0.367188,0.570281,0.194033,0.485719,0.560275,0.496720,0.523329
053_S_4557,0.437899,0.371801,0.390647,0.601737,0.820094,0.628986,0.550265,0.727898,0.516281,0.509992,...,0.740470,0.368328,0.279656,0.485633,0.578782,0.205656,0.233461,0.788992,0.801559,0.494200
073_S_4300,0.507318,0.516599,0.447872,0.234254,0.473863,0.207288,0.537699,0.782304,0.510855,0.475166,...,0.699437,0.397107,0.412106,0.445935,0.645697,0.328136,0.440785,0.663687,0.646746,0.617501
041_S_4014,0.508396,0.568496,0.430380,0.640320,0.799409,0.490875,0.451155,0.723914,0.373445,0.429058,...,0.426977,0.425292,0.238418,0.377493,0.324971,0.263497,0.162345,0.689898,0.487404,0.184313


In [4]:
patient_kg_path = "./data/KG/patient_kg.pkl"

# load graph
with open(patient_kg_path, 'rb') as f:
    G = pickle.load(f) # patient_kg


ad_node = 'path(MESH:"Alzheimer Disease")'
patient_to_ad_length = []
for node, attr in G.nodes(data=True):
    if attr['label'] == 'Patient':
        if nx.has_path(G, node, ad_node):
            spl = nx.shortest_path_length(G, node, ad_node)
            patient_to_ad_length.append(spl)
        else:
            patient_to_ad_length.append(None)


map patients onto KG

In [5]:
def extract_hgnc(node_id):
    match = re.search(r'HGNC:"([^"]+)"\)', node_id)
    return match.group(1) if match else ""

def add_patient_to_kg(kg:nx.MultiDiGraph, exp:pd.DataFrame, output_filename:str):
    """Add patients to KG with gene-expression-value as edge_weight, 
    create edges between patients and all overlapping HGNC-proteins in KG and expression data .

    Args:
        kg (nx.MultiDiGraph): knowledge graph
        exp (pd.DataFrame): patient gene expression data (num_patients x num_genes)
        output_filename(str): save graph

    Returns:
        nx.MultiDiGraph: A new Knowledge Graph with patient nodes
    """
    # normalize expression data
    exp_log = np.log1p(exp)
    #exp_zscore = (exp_log - exp_log.mean())/exp_log.std()
    exp_minmax = (exp_log - exp_log.min())/(exp_log.max()-exp_log.min())
    
    exp_genes = list(exp.columns)
    print('The number of genes in Gene Expression data:',len(exp_genes))
    patients = list(exp.index)
    print('The number of patients:', len(patients))

    # add patients to kg
    G = kg.copy()
    kg_proteins = [node for node in G.nodes(data=True) if node[1]['label'] == 'Protein']
    print('The number of KG proteins: ',len(kg_proteins))

    mapped_nodes = set()
    for i in tqdm(range(len(patients)), desc='Add patients to KG'):
        head = patients[i]
        target = labels[i]
        if head not in G.nodes:
            G.add_node(head, label='Patient', y=target)

        for node,attr in kg_proteins:
            
            gene_name = extract_hgnc(node) # only link to HGNC proteins

            if gene_name in exp_genes:
                j = exp_genes.index(gene_name)
                #print(head)
                #print(gene_name)
                exp_value = exp.iloc[i,j]
                normalized_value = exp_minmax.iloc[i,j]
                #print(exp_value)
                G.add_edge(head, node, 'express', edge_weight = normalized_value)
                G.add_edge(node, head, 'rev_express', edge_weight = normalized_value)
                mapped_nodes.add(node)
    print(f'The number of mapped protein-nodes is {len(mapped_nodes)}')
    
    # save graph
    with open(output_filename, 'wb') as f:
        pickle.dump(G, f)
    print('-------------------------- Done ------------------------------')
    return G, mapped_nodes

In [92]:
G, mapped_nodes = add_patient_to_kg(kg, exp, './data/KG/patient_kg.pkl')

The number of genes in Gene Expression data: 19100
The number of patients: 744
The number of KG proteins:  1301


Add patients to KG: 100%|██████████| 744/744 [03:50<00:00,  3.22it/s]


The number of mapped protein-nodes is 735
-------------------------- Done ------------------------------


## Prepare HeteroData

(1) Get a AD relevance score for each node
+ cosine similarity / biliner similarity between each node and AD-node;
  + cosine similarity: $ cos(X,y) = \frac{X \cdot y}{||X|| \cdot ||y||} $
  + bilinear similarity: add a light learnable weight matrix to scale cosine similarity, such that it can learn the similarity in a AD-disease-aware way. $ bil(X,y) = X^T \cdot W \cdot y + b $
   
+ graph-aware relavence propagation
  + the similarity scores only capture node-level relevant, a following propagation is better to be applied to get a netwrok-level relevance --> relevant nodes' neighborhood are also relevant;
  + can train a edge-type-aware relevance model, but i don't think necessary.

a. cosine similarity

In [5]:
with open('./data/hgt_kge_node_mappings.pkl', 'rb') as f:
    node_mappings = pickle.load(f)
kg_embed = torch.load('./data/hgt_KGEmbeddings.pt')


# compute cosine similarity to ad_embed
def get_cosine_similarity(kg_embed:dict, node_mappings:dict) -> Tuple[Dict[str,torch.Tensor],Dict[str,torch.Tensor]]:
    """calculate cosine similarity between all nodes in KG and AD-node.

    Args:
        kg_embed (dict): knowledge graph embeddings
        node_mappings (dict): node mappings

    Returns:
        Dict[str,List]: contains cosine similarity scores
    """
    ad_idx = node_mappings['Pathology']['path(MESH:"Alzheimer Disease")']
    ad_embed = kg_embed['Pathology'][ad_idx]

    node_relevances_list = {}
    all_similarities = []
    for node_type, node_info in node_mappings.items():
        
        node_relevances_list[node_type] = []
        for node_name, node_idx in node_info.items():
            node_embed = kg_embed[node_type][node_idx]
            score = F.cosine_similarity(node_embed, ad_embed, dim=0)
            all_similarities.append(score)
            
            node_relevances_list[node_type].append(score)
    
    # MinMax normalization of similarities
    min_sim = min(all_similarities)
    max_sim = max(all_similarities)
    new_relevance = {}
    for nt, scores in node_relevances_list.items():
        scores = torch.tensor(scores)
        scores = (scores - min_sim)/(max_sim - min_sim)
        new_relevance[nt] = scores
 
    normalized_relevance = {nt:torch.tensor(v) for nt, v in new_relevance.items()}
    node_relevances_list = {nt:torch.tensor(v) for nt, v in node_relevances_list.items()}
    return normalized_relevance, node_relevances_list

node_relevances, origin_node_relevance = get_cosine_similarity(kg_embed, node_mappings)

/var/folders/ft/2nftg5q91n5dhpf5dtts4gkm0000gn/T/ipykernel_90901/1663247335.py:41: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  normalized_relevance = {nt:torch.tensor(v) for nt, v in new_relevance.items()}


+ shortest path to AD as relevance score


In [44]:
def get_shortest_path(kg:nx.MultiDiGraph, target_node: str = 'path(MESH:"Alzheimer Disease")'):
    
    #ad_node = 'path(MESH:"Alzheimer Disease")'
    shortest_paths = {}
    for node, attr in kg.nodes(data=True):
        node_type = attr['label']
        if node_type not in shortest_paths:
            shortest_paths[node_type] = []
        #print(node)
        #print(ad_node)
        #break
        if nx.has_path(kg, node, target_node):
            shortest_path = nx.shortest_path_length(kg, node, target_node)
            shortest_paths[node_type].append(shortest_path)
        else:
            shortest_paths[node_type].append(None)
    return shortest_paths

shortest_paths = get_shortest_path(kg)

In [46]:
def get_node_relevance(kg,kg_embed, node_mappings, method:str, output_type:str='list'):

    if method == 'shortest path':
        node_relevances = get_shortest_path(kg)
    elif method == 'cosine similarity':
        node_relevances, _ = get_cosine_similarity(kg_embed, node_mappings)
    else:
        raise ValueError('Invalid input of method.')
    
    return node_relevances


In [51]:
def add_attributes_to_graph(G:nx.MultiDiGraph, node_relevances:Dict, patient_labels, output_filename:str):
    
    # add relevance score to nodes in patient_graph
    for nt, scores in node_relevances.items():
        i = 0
        for node, attr in G.nodes(data=True):
            if attr['label'] == nt:
                if 'ad_relevance' not in attr:
                    attr['ad_relevance'] = scores[i]
                i += 1

            if i == len(scores):
                continue
    # add patient node y_labels to node attributes
    j = 0
    for node, attr in G.nodes(data=True):
        if attr.get('label') == 'Patient':
            attr['y'] = patient_labels[j]
            j += 1
    
    # save graph
    with open(output_filename, 'wb') as f:
        pickle.dump(G, f)
    print('-------------------------- Done ------------------------------')
    return G

G = add_attributes_to_graph(G,node_relevances,labels, './data/KG/patient_kg.pkl')

-------------------------- Done ------------------------------


In [52]:
for n,attr in G.nodes(data=True):
    print(n, attr)

g(HGNC:"BDNF") {'label': 'Gene', 'pure': None, 'species': 9606.0, 'name': 'BDNF', 'involved_genes': 'BDNF', 'namespace': 'HGNC', 'type': 'Gene', 'ad_relevance': tensor(0.2721)}
g(HGNC:"TP53") {'label': 'Gene', 'pure': None, 'species': 9606.0, 'name': 'TP53', 'involved_genes': 'TP53', 'namespace': 'HGNC', 'type': 'Gene', 'ad_relevance': tensor(0.3373)}
g(HGNC:"PPP1CC") {'label': 'Gene', 'pure': None, 'species': 9606.0, 'name': 'PPP1CC', 'involved_genes': 'PPP1CC', 'namespace': 'HGNC', 'type': 'Gene', 'ad_relevance': tensor(0.3298)}
g(HGNC:"VLDLR") {'label': 'Gene', 'pure': None, 'species': 9606.0, 'name': 'VLDLR', 'involved_genes': 'VLDLR', 'namespace': 'HGNC', 'type': 'Gene', 'ad_relevance': tensor(0.2714)}
g(HGNC:"SLC1A2") {'label': 'Gene', 'pure': None, 'species': 9606.0, 'name': 'SLC1A2', 'involved_genes': 'SLC1A2', 'namespace': 'HGNC', 'type': 'Gene', 'ad_relevance': tensor(0.3010)}
g(HGNC:"PTK2B") {'label': 'Gene', 'pure': None, 'species': 9606.0, 'name': 'PTK2B', 'involved_genes'

b. bilinear similarity

In [ ]:
# already have shgt_embeddings, can use cosine similarity as relevance score
class BilinearSimilarity(nn.Module):
    def __init__(self, embedding_1_dim, embedding_2_dim):
        super().__init__()
        self.W = nn.Parameter(torch.randn(embedding_1_dim, embedding_2_dim))
        self.bias = nn.Parameter(torch.Tensor(1))

    def forward(self, node_emb, ad_emb):
        """
        node_emb: [N, d]
        ad_emb:   [d] or [1, d]
        """
        if ad_emb.dim() == 1:
            ad_emb = ad_emb.unsqueeze(0)  # [1, d]

        # h_v^T W h_AD
        scores = torch.matmul(node_emb, self.W) @ ad_emb.T  # [N, 1]
        scores = scores.squeeze(-1)

        return torch.sigmoid(scores)  # relevance in (0,1)

class BiLinearSimilarity(nn.Module):
 
    def __init__(self, tensor_1_dim, tensor_2_dim, activation=None):
        super(BiLinearSimilarity, self).__init__()
        self.weight_matrix = nn.Parameter(torch.Tensor(tensor_1_dim, tensor_2_dim))
        self.bias = nn.Parameter(torch.Tensor(1))
        self.activation = activation
        self.reset_parameters()
 
    def reset_parameters(self):
        nn.init.xavier_uniform_(self.weight_matrix)
        self.bias.data.fill_(0)
 
    def forward(self, tensor_1, tensor_2):
        intermediate = torch.matmul(tensor_1, self.weight_matrix)
        result = (intermediate * tensor_2).sum(dim=-1) + self.bias
        if self.activation is not None:
            result = self.activation(result)
        return result

# need to mannualy assign labels to nodes in KG 
# (based on edge-distance to AD? ---- I think i will take this one
# or based on prior knowledge? ) -> at most combine this one as well.
def train_bilinear(
    model,
    node_emb,
    ad_emb,
    labels,
    optimizer,
    epochs=200
):
    """
    labels: [N] ∈ {0,1}
    """
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()

        scores = model(node_emb, ad_emb)
        loss = F.binary_cross_entropy(scores, labels.float())

        loss.backward()
        optimizer.step()

        if epoch % 20 == 0:
            print(f"Epoch {epoch:03d} | Loss {loss.item():.4f}")


c. graph-aware relavence propagation

In [ ]:
def relevance_propagation(
    edge_index,
    init_relevance,
    num_nodes,
    alpha=0.85,
    num_iters=20
):
    """
    edge_index: [2, E]
    init_relevance: [N] in (0,1)
    """
    r0 = init_relevance
    r = r0.clone()

    # Build degree-normalized adjacency
    row, col = edge_index
    deg = torch.bincount(row, minlength=num_nodes).float()
    deg_inv = 1.0 / (deg + 1e-6)

    for _ in range(num_iters):
        r_new = torch.zeros_like(r)

        # message passing: r_j -> r_i
        r_new.index_add_(
            0,
            row,
            r[col] * deg_inv[col]
        )

        r = alpha * r_new + (1 - alpha) * r0

    return r


(2) add edge_weight to graph as cosine similarity between embeddings

In [14]:
def get_edge_weights(G:nx.MultiDiGraph, kg_embed:Dict, node_mappings:Dict, node_relevances, type:str='relevance'):
    """edge_weight between nodes: 
       - patient ~ protein: gene expression value
       - other edges: (1) cosine similarity between embeddings; 
                      (2) average of nodes' relevance score;
                      
    Args:
        G (nx.MultiDiGraph): graph with patient nodes
        kg_embed (dict): embeddings of knowledge graph nodes
        node_mappings (dict): _description_
        type (str): choose between 'cosine' and 'relevance'. 
                    to set non-patient-nodes edge_weight.

    Returns:
        Dict[Tuple[str, str, str], list]: edge weights
    """
    edge_weight_list:Dict[Tuple[str, str, str], list] = {}
    for source, target, rel_type, edge_attr in G.edges(data=True, keys=True):
        
        src_type = G.nodes[source]["label"]
        dst_type = G.nodes[target]["label"]
        if src_type != 'Patient' and dst_type != 'Patient':
            # get source and target embeddings
            src_idx = node_mappings[src_type][source]
            dst_idx = node_mappings[dst_type][target]
            
            src_embed = kg_embed[src_type][src_idx]
            dst_embed = kg_embed[dst_type][dst_idx]
            
            # calculate cosine similarity
            if type == 'cosine':
                score = F.cosine_similarity(src_embed, dst_embed, dim=0)
                edge_type_tuple = (src_type, str(rel_type), dst_type)
                if edge_type_tuple not in edge_weight_list:
                    edge_weight_list[edge_type_tuple] = []
            
            # claculate average of node_relevance score
            elif type == 'relevance':
                src_rs = node_relevances[src_type][src_idx]
                dst_rs = node_relevances[dst_type][dst_idx]
                score = (src_rs + dst_rs)/2
                
                edge_type_tuple = (src_type, str(rel_type), dst_type)
                if edge_type_tuple not in edge_weight_list:
                    edge_weight_list[edge_type_tuple] = []
            else:
                raise ValueError('Invalid input of type. ["cosine", "relevance"]')
                
            edge_weight_list[edge_type_tuple].append(score)
        
        if src_type == 'Patient' or dst_type == 'Patient':
            # get weight
            weight = torch.tensor(edge_attr['edge_weight'])
            edge_type_tuple = (src_type, str(rel_type), dst_type)
            if edge_type_tuple not in edge_weight_list:
                edge_weight_list[edge_type_tuple] = []
            #print(edge_type_tuple)
            edge_weight_list[edge_type_tuple].append(weight)
        
    return edge_weight_list

In [15]:
edge_weight_list = get_edge_weights(G, kg_embed, node_mappings, node_relevances)

In [16]:
edge_weight_list

{('Gene', 'decreases', 'Protein'): [tensor(0.3101),
  tensor(0.3869),
  tensor(0.3407),
  tensor(0.3472),
  tensor(0.3455)],
 ('Gene', 'increases', 'Gene'): [tensor(0.3028)],
 ('Gene', 'increases', 'Protein'): [tensor(0.2559),
  tensor(0.2324),
  tensor(0.2956),
  tensor(0.4053),
  tensor(0.2598),
  tensor(0.2727),
  tensor(0.3161),
  tensor(0.3843),
  tensor(0.3833),
  tensor(0.3799),
  tensor(0.3447),
  tensor(0.2954),
  tensor(0.2727),
  tensor(0.2752),
  tensor(0.3121),
  tensor(0.2439),
  tensor(0.2727),
  tensor(0.2712),
  tensor(0.3012),
  tensor(0.2724),
  tensor(0.3198),
  tensor(0.3148),
  tensor(0.2946),
  tensor(0.2674),
  tensor(0.2674),
  tensor(0.2706)],
 ('Gene', 'rev_association', 'Gene'): [tensor(0.2889), tensor(0.2605)],
 ('Gene', 'rev_association', 'Protein'): [tensor(0.2709),
  tensor(0.3159),
  tensor(0.2911),
  tensor(0.2911),
  tensor(0.3630),
  tensor(0.3613),
  tensor(0.3379),
  tensor(0.2967),
  tensor(0.2955),
  tensor(0.2865),
  tensor(0.2955),
  tensor(0.3

In [8]:
def create_train_val_test_masks(num_patients, train_ratio=0.8, val_ratio=0.1):
    total_size = num_patients
    train_size, val_size, test_size = train_ratio*total_size, val_ratio* total_size, (1-train_ratio - val_ratio)*total_size

    train_mask = [i < train_size for i in range(total_size)]
    val_mask = [train_size <= i < train_size + val_size for i in range(total_size)]
    test_mask = [i >= train_size + val_size for i in range(total_size)]
    
    return train_mask, val_mask, test_mask

(3) convert nx.Multigraph to HeteroData
+ data.x
+ data.y
+ data.edge_index
+ data.edge_weight: 
+ data.relevance
+ data.train_mask
+ data.val_mask
+ data.test_mask
+ data.topo_features (optional)

In [9]:
def networkx_to_hetero_data(graph: nx.MultiDiGraph) -> Tuple[HeteroData, Dict[str, Dict[Any, int]]]:
    """Convert a NetworkX heterogeneous graph to HeteroData."""
    data = HeteroData()
    node_mappings: Dict[str, Dict[Any, int]] = {}
    node_relevances = {}
    labels = []

    for node_id, attrs in graph.nodes(data=True):
        node_type = attrs.get("label")
        if node_type not in node_mappings:
            node_mappings[node_type] = {}
            node_relevances[node_type] = []
        if node_id not in node_mappings[node_type]:
            node_mappings[node_type][node_id] = len(node_mappings[node_type])
            node_relevances[node_type].append(attrs.get('ad_relevance'))
        if node_type == 'Patient':
            labels.append(attrs.get('y'))
    # add data attributes
    #data.relevance
    
    for node_type, mapping in node_mappings.items():
        data[node_type].num_nodes = len(mapping)
        if node_type != 'Patient':
            data[node_type].relevance = torch.tensor(node_relevances[node_type])
    print(f"Found {len(node_mappings)} node types.")
    
    # data.y
    data['Patient'].y = torch.tensor(labels, dtype=torch.int)
    # creat masks
    train_mask, val_mask, test_mask = create_train_val_test_masks(data["Patient"].num_nodes)
    data['Patient'].train_mask=torch.tensor(train_mask, dtype=torch.bool)
    data['Patient'].val_mask=torch.tensor(val_mask, dtype=torch.bool)
    data['Patient'].test_mask=torch.tensor(test_mask, dtype=torch.bool)
    print("Add y label and train, val, test mask to data['Patient].")

    # edge_index
    edge_lists: Dict[Tuple[str, str, str], list] = {}
    edge_weight_list = {}
    for source, target, rel_type, edge_attr in graph.edges(data=True, keys=True):
        src_type = graph.nodes[source]["label"]
        dst_type = graph.nodes[target]["label"]
        edge_type_tuple = (src_type, str(rel_type), dst_type)
        if edge_type_tuple not in edge_lists:
            edge_lists[edge_type_tuple] = []
        edge_lists[edge_type_tuple].append(
            [
                node_mappings[src_type][source],
                node_mappings[dst_type][target],
            ]
        )
        if src_type == 'Patient' or dst_type == 'Patient':
            # get weight
            weight = edge_attr['edge_weight']
            if edge_type_tuple not in edge_weight_list:
                edge_weight_list[edge_type_tuple] = []
            
            edge_weight_list[edge_type_tuple].append(weight)
        

    for edge_type_tuple, edges in edge_lists.items():
        data[edge_type_tuple].edge_index = (
            torch.tensor(edges, dtype=torch.long).t().contiguous()
        )
    print(f"Found {len(edge_lists)} edge types.")

    # edge_weight
    for edge_type_tuple, edge_weights in edge_weight_list.items():
        data[edge_type_tuple].edge_weight = torch.tensor(edge_weights)
    print("Add (patient,rel,protein) edge_weights to HeteroData")

    print("Conversion complete!")
    return data, node_mappings

In [10]:
data, new_node_mappings = networkx_to_hetero_data(G)

Found 16 node types.
Add y label and train, val, test mask to data['Patient].
Found 996 edge types.
Add (patient,rel,protein) edge_weights to HeteroData
Conversion complete!


In [11]:
data

HeteroData(
  Gene={
    num_nodes=137,
    relevance=[137],
  },
  Abundance={
    num_nodes=408,
    relevance=[408],
  },
  BiologicalProcess={
    num_nodes=446,
    relevance=[446],
  },
  Activity={
    num_nodes=546,
    relevance=[546],
  },
  Pathology={
    num_nodes=126,
    relevance=[126],
  },
  MicroRna={
    num_nodes=46,
    relevance=[46],
  },
  Protein={
    num_nodes=1301,
    relevance=[1301],
  },
  Rna={
    num_nodes=111,
    relevance=[111],
  },
  Translocation={
    num_nodes=64,
    relevance=[64],
  },
  Reaction={
    num_nodes=13,
    relevance=[13],
  },
  Degradation={
    num_nodes=56,
    relevance=[56],
  },
  CellSecretion={
    num_nodes=30,
    relevance=[30],
  },
  CellSurfaceExpression={
    num_nodes=3,
    relevance=[3],
  },
  Complex={
    num_nodes=373,
    relevance=[373],
  },
  Composite={
    num_nodes=72,
    relevance=[72],
  },
  Patient={
    num_nodes=744,
    y=[744],
    train_mask=[744],
    val_mask=[744],
    test_mask=[744]

### Model define

+ ConditioanlMessageLayer:
    + Conditional message passing:
    + patient - protein edge:
      + $ r_1(v) $ = relevance score of non-patient node;
      + $ r_2(u) = \begin{cases} 1 & \text{if }  y_u = 1 \\ 0 & \text{if }  y_u = 0 \end{cases}$, where $u$ is patient ndoe;
      + $ r_3(u,v) = e_{u,v}$, the edge_weight between node $u$ and $v$;
      + combined gating rule:
        + $ g_{u,v} = \sigma(\alpha (r_1(v)*r_3(u,v) - \theta)) * r_2(u) $; 
        + $ g_{u,v} \in [0,1]$
    + all other edges gating:
      + $g_{u,v}$ = edge weight
        + edge_weight = embedding similarity between nodes;(`current implementation`)
        + edge_weight = the average node_relevance of two nodes;(`prefer`)
        + or edge_weight = 1: `not good because gated_edge_weight of edge_patient&protein < 1`
  
+ GNNModel
  + instead of combining gate-layer and GNN layer before the final classifier layer, here the rule-layer should be incorporated in each GNN layer;
  + incorporate gated_edge_weight to GAT layer.
+ `==> edge_weights are changing during training, should not do this `

In [143]:
class ConditionalRuleLayer(nn.Module):
    """
    Computes edge-level gates based on: node relevance; patient disease label; edge weight
    """
    def __init__(self):
        super().__init__()
        self.theta = nn.Parameter(torch.zeros(1))
        self.alpha = nn.Parameter(torch.ones(1))

    def forward(
        self,
        dst_relevance,  
        edge_weight,    
        src_is_patient: bool, 
        is_ad_patient: bool        
    ):
        """
        Returns:
            gate: [E] ∈ (0,1)
        """
        base = dst_relevance * edge_weight
        score = torch.sigmoid(self.alpha * (base - self.theta))

        gate = torch.ones_like(score)
        #print('all score: ',score)
        #print('ad_edge score:', score[src_is_patient & is_ad_patient])
        #print('healthy edges score:', 1.0 - score[src_is_patient & ~is_ad_patient])

        # Non-patient-node → AD patient
        gate[src_is_patient & is_ad_patient] = score[src_is_patient & is_ad_patient]

        # Non-patient → Control patient
        gate[src_is_patient & ~is_ad_patient] = 1.0 - score[src_is_patient & ~is_ad_patient]

        return gate

In [ ]:
class ConditionHeteroGAT(torch.nn.Module):
    """Heterogeneous GNN encoder with learnable node embeddings."""
    def __init__(self, data, in_channels, hidden_channels, out_channels, 
                 dropout_rate, heads, aggr='sum', num_layers = 2):
        super().__init__()
        
        self.data = data
        self.num_layers = num_layers
        self.dropout_rate = dropout_rate
        self.hidden_channels = hidden_channels
        self.out_channels = out_channels

        # GAT backbone
        self.convs = nn.ModuleList()
        self.convs.append(HeteroConv({
            edge_type: GATConv(in_channels, hidden_channels, heads=heads, add_self_loops=False) for edge_type in data.edge_types
        }, aggr=aggr))
        for _ in range(num_layers - 1):
            self.convs.append(HeteroConv({
            edge_type: GATConv(hidden_channels, hidden_channels, heads=1, add_self_loops=False) for edge_type in data.edge_types
        }, aggr=aggr))
        
            # batch normalization?

        # Conditional rules
        self.conditional_rule = ConditionalRuleLayer()

        # final layer
        self.convs.append(HeteroConv({
            edge_type: GATConv(hidden_channels, out_channels, heads=1, add_self_loops=False) for edge_type in data.edge_types
        }, aggr=aggr))



    def forward(self, x_dict, edge_index_dict, edge_weight_dict, relevance_dict, y_labels):
        
        gated_edge_weight_dict = {}

        for edge_type in edge_index_dict:
            src_type, _, dst_type = edge_type
            edge_index = edge_index_dict[edge_type]
            edge_weight = torch.tensor(edge_weight_dict[edge_type])
            #print('edge_weight:',edge_weight)
            
            # Default: no gating
            gated_weight = edge_weight

            if src_type == "Patient" and dst_type != "Patient":
                src_nodes = edge_index[0]
                #print('src_nodes index:',src_nodes)
                dst_nodes = edge_index[1]

                dst_relevance = torch.tensor([relevance_dict[dst_type][dst_node] for dst_node in dst_nodes])
                #print('dst_relevance:', dst_relevance)
                src_labels = torch.tensor([y_labels[src_node] for src_node in src_nodes])
                #print('src_labels:', src_labels)
                
                gate = self.conditional_rule(
                    dst_relevance=dst_relevance,
                    edge_weight=edge_weight,
                    src_is_patient=torch.ones_like(src_labels, dtype=torch.bool),
                    is_ad_patient=(src_labels == 1)
                )

                gated_weight = edge_weight * gate
            
            if src_type != "Patient" and dst_type == "Patient":
                src_nodes = edge_index[0]
                #print('src_nodes index:',src_nodes)
                dst_nodes = edge_index[1]

                src_relevance = torch.tensor([relevance_dict[src_type][src_node] for src_node in src_nodes])
                #print('dst_relevance:', dst_relevance)
                dst_labels = torch.tensor([y_labels[dst_node] for dst_node in dst_nodes])
                #print('src_labels:', src_labels)
                
                #print('dst_is_patient:',torch.ones_like(dst_labels, dtype=torch.bool))
                
                conditional_rule = ConditionalRuleLayer()
                gate = conditional_rule(
                    dst_relevance=src_relevance,
                    edge_weight=edge_weight,
                    src_is_patient=torch.ones_like(dst_labels, dtype=torch.bool),
                    is_ad_patient=(dst_labels == 1)
                )

                gated_weight = edge_weight * gate
            
            gated_edge_weight_dict[edge_type] = gated_weight
        
        # HeteroGAT 
        for conv in self.convs:
            x_dict = conv(x_dict, edge_index_dict, gated_edge_weight_dict)
            x_dict = {key: F.relu(x) for key, x in x_dict.items()}
            x_dict = {key: F.dropout(x, p=self.dropout_rate, training=self.training) for key, x in x_dict.items()}
            

        return x_dict


In [ ]:
# prepare input: edge_index_dict, ed
print('edge_index_dict')
edge_index_dict = {
        edge_type: data[edge_type].edge_index.cpu()
        for edge_type in data.edge_types
    }
print('\nedge_weight_dict')
edge_weight_dict = edge_weight_list
print('\nrelevance dict')
relevance_dict = node_relevances


edge_index_dict

edge_weight_dict

relevance dict


In [145]:
gated_edge_weight_dict = {}

for edge_type in edge_index_dict:
    src_type, _, dst_type = edge_type
    edge_index = edge_index_dict[edge_type]
    edge_weight = torch.tensor(edge_weight_dict[edge_type])
    #print('edge_weight:',edge_weight)
    
    # Default: no gating
    gated_weight = edge_weight

    if src_type == "Patient" and dst_type != "Patient":
        src_nodes = edge_index[0]
        #print('src_nodes index:',src_nodes)
        dst_nodes = edge_index[1]

        dst_relevance = torch.tensor([relevance_dict[dst_type][dst_node] for dst_node in dst_nodes])
        #print('dst_relevance:', dst_relevance)
        src_labels = torch.tensor([labels[src_node] for src_node in src_nodes])
        #print('src_labels:', src_labels)
        
        conditional_rule = ConditionalRuleLayer()
        gate = conditional_rule(
            dst_relevance=dst_relevance,
            edge_weight=edge_weight,
            src_is_patient=torch.ones_like(src_labels, dtype=torch.bool),
            is_ad_patient=(src_labels == 1)
        )

        gated_weight = edge_weight * gate
        print(edge_type, gate)
    
    if src_type != "Patient" and dst_type == "Patient":
        src_nodes = edge_index[0]
        #print('src_nodes index:',src_nodes)
        dst_nodes = edge_index[1]

        src_relevance = torch.tensor([relevance_dict[src_type][src_node] for src_node in src_nodes])
        #print('dst_relevance:', dst_relevance)
        dst_labels = torch.tensor([labels[dst_node] for dst_node in dst_nodes])
        #print('src_labels:', src_labels)
        
        #print('dst_is_patient:',torch.ones_like(dst_labels, dtype=torch.bool))
        
        conditional_rule = ConditionalRuleLayer()
        gate = conditional_rule(
            dst_relevance=src_relevance,
            edge_weight=edge_weight,
            src_is_patient=torch.ones_like(dst_labels, dtype=torch.bool),
            is_ad_patient=(dst_labels == 1)
        )

        gated_weight = edge_weight * gate
        print(edge_type, gate)
    
    gated_edge_weight_dict[edge_type] = gated_weight

('Protein', 'rev_express', 'Patient') tensor([0.5030, 0.5028, 0.4970,  ..., 0.3203, 0.6718, 0.3191],
       dtype=torch.float64, grad_fn=<IndexPutBackward0>)
('Patient', 'express', 'Protein') tensor([0.5030, 0.4995, 0.5241,  ..., 0.4949, 0.5896, 0.3191],
       dtype=torch.float64, grad_fn=<IndexPutBackward0>)


### Conditional Gating only before final layer
- similar to FireGNN, the former layers are normal GNN but add a Gating layer to incorporate Conditional rules;
- this way only uses node-relevance and edge-weight (gene expression value) to decide the strength of ad-relevance;
- expand rule-features from node-level to neighborhood level:
  - one-hop relevance intensity: relevance* expression_value
  - eg. 2-hop relevance intensity: because the connected proteins are the same, the 2-hop relevance will also be the same to all patients.
  - `need more features here to help decide disease-relevant-intensity`
- Not using patient labels to directly affect message passing, but by designing fuzzy rules to detect disease-like behavior and in turn passing more disease-specific knowledge to disease patients while less to controls.
- also train the model by link prediction loss and classification loss.

In [12]:
def compute_rule_features(data,relevance_threshold:float=0.05):

    patient_idx, protein_idx = data[('Patient', 'express', 'Protein')].edge_index
    edge_weights = data[('Patient', 'express', 'Protein')].edge_weight # Expression values
    num_patients = data['Patient'].num_nodes
    node_relevance_dict = {nt:data[nt].relevance for nt in data.node_types if nt != 'Patient'}
    
    # 1. Intensity: Sum(Expression * Relevance)
    prot_rel = node_relevance_dict['Protein'][protein_idx]
    # print('maximum protein relevance: ', max(prot_rel)), 0.2004
    # print('minimum protein relevance: ', min(prot_rel)), -0.1284
    intensity = scatter_sum(edge_weights * prot_rel, patient_idx, dim=0)
    print(intensity.size())

    # 2. Max-Relevance Signal: max(Exp * Rel)
    # captures the "strongest biomarker" signal in the neighborhood
    prot_rel = prot_rel = node_relevance_dict['Protein'][protein_idx]
    combined_signal = edge_weights * prot_rel
    max_rel_signal, _ = scatter_max(combined_signal, patient_idx, dim=0, dim_size=num_patients)

    
    # Stack into a [num_patients, 3] feature matrix
    # Handle patients with no edges (padding with zeros)
    features = torch.zeros((num_patients, 2))
    features[:intensity.size(0), 0] = intensity
    features[:max_rel_signal.size(0), 1] = max_rel_signal
    
    return features


In [13]:
relevance_features = compute_rule_features(data)
relevance_features

torch.Size([744])


tensor([[87.1624,  0.2694],
        [78.3089,  0.3121],
        [87.2475,  0.2565],
        ...,
        [88.7893,  0.2770],
        [93.6007,  0.3132],
        [93.4231,  0.2967]])

In [18]:
class FuzzyRuleLayer(nn.Module):
    def __init__(self, num_rules=5):
        super().__init__()
        # alpha: sharpness of the rule, theta: threshold for "significance"
        self.theta = nn.Parameter(torch.zeros(num_rules))
        self.alpha = nn.Parameter(torch.ones(num_rules))

    def forward(self, relevance_features):
        # relevance_features shape: [num_patients, num_rules]
        # Eq (3) from paper: r = sigmoid(alpha * (f - theta))
        r = torch.sigmoid(self.alpha * (relevance_features - self.theta))
        return r

class HeteroFireGNN(nn.Module):
    def __init__(self, data, hidden_channels, out_channels, heads, dropout_rate, num_rules=5):
        super().__init__()

        self.data = data
        self.dropout_rate = dropout_rate
        self.hidden_channels = hidden_channels
        self.out_channels = out_channels

        # 1. Neural Component
        # Use HeteroConv to handle different edge types differently
        self.embeddings = torch.nn.ModuleDict({
            node_type: torch.nn.Embedding(num_nodes, hidden_channels)
            for node_type, num_nodes in {nt: data[nt].num_nodes for nt in data.node_types}.items()
        })
        self.conv1 = (HeteroConv({
            edge_type: GATConv((-1, -1), hidden_channels, heads, add_self_loops=False)
            for edge_type in data.edge_types
        }))
        self.conv2 = (HeteroConv({
            edge_type: GATConv(hidden_channels*heads, hidden_channels, heads=1, add_self_loops=False)
            for edge_type in data.edge_types
        }))
        
        # 2. Symbolic Component: Fuzzy Rules for Patients
        self.rule_layer = FuzzyRuleLayer(num_rules=num_rules)
        self.gate = nn.Linear(num_rules, 1)

        # 3. classifier
        self.classifier = nn.Sequential(
            nn.Linear(hidden_channels * 2, hidden_channels),
            nn.ReLU(),
            nn.Linear(hidden_channels, out_channels)
        )

        # 3. Link Prediction Head 
        self.rel_weights = nn.ParameterDict({
            "__".join(edge_type): nn.Parameter(torch.ones(hidden_channels))
            for edge_type in data.edge_types
        })
        for param in self.rel_weights.values():
            torch.nn.init.xavier_uniform_(param.unsqueeze(0))

    def forward(self,edge_index_dict, patient_rule_features):
        """
        x_dict: Dictionary of node features for all types
        edge_index_dict: Dictionary of edges
        patient_rule_features: The [N, 3] matrix from compute_biomedical_rule_features
        """
        # --- Neural Phase ---
        # Pass messages across the whole KG
        x_dict = {
                node_type: embedding.weight
                for node_type, embedding in self.embeddings.items()
            }

        h_dict = self.conv1(x_dict, edge_index_dict)
        h_dict = {key: F.elu(v) for key, v in h_dict.items()}
        h_dict = {key: F.dropout(v, p=self.dropout_rate, training=self.training) for key, v in h_dict.items()}
        h_dict = self.conv2(h_dict, edge_index_dict)
        # Target the patient embeddings
        h_patient = h_dict['Patient']
        h_protein = h_dict['Protein']
        
        # --- Symbolic Phase ---
        # apply FireGNN Fuzzy Logic: r = sigmoid(alpha * (f - theta))
        # this checks if the patient's neighborhood satisfies disease logic
        r_symbolic = self.rule_layer(patient_rule_features) # -> [num_patients, num_features]
        
        # gives each patient a score (learnt from relevance features) 
        # due to trained by classification, disease_patients have high score while controls have low.
        r_gate = torch.sigmoid(self.gate(r_symbolic)) # -> [num_patients, 1]
        
        pat_idx, prot_idx = edge_index_dict[('Patient', 'express', 'Protein')]
        # expand r_gate to the same dimension of protein_embedding
        edge_gates = r_gate[pat_idx] # -> [num_patient_protein_edges, 1]

        # edge_weight (exp_value) should also be involved
        # here the protein_embeddings are scaled by learned rule_gate score;
        gated_messages = h_protein[prot_idx] * edge_gates  # -> [num_patient_protein_edges, 1]
        
        # aggregate gated messages -> 
        selective_message = scatter_sum(gated_messages, pat_idx, dim=0, dim_size=h_patient.size(0))
        
        # --- Fusion ---
        # Combine global GNN context (h_patient) and selective context
        combined = torch.cat([h_patient, selective_message], dim=1)
        out = self.classifier(combined)

        return h_dict, F.log_softmax(out, dim=1), edge_gates
    
    def decode(self, h_dict, edge_index, edge_type):
        """Link Prediction Decoder: Predicts probability of edges"""
        u_type, rel, v_type = edge_type
        src, dst = edge_index
        
        h_src = h_dict[u_type][src]
        h_dst = h_dict[v_type][dst]
        
        rel_key = "__".join(edge_type)
        rel_w = self.rel_weights[rel_key]
        return (h_src * rel_w * h_dst).sum(dim=-1)

In [19]:
model = HeteroFireGNN(
    data=data,
    hidden_channels=128,
    out_channels=2,
    heads=2,
    dropout_rate=0.5,
    num_rules=2
)
model

HeteroFireGNN(
  (embeddings): ModuleDict(
    (Gene): Embedding(137, 128)
    (Abundance): Embedding(408, 128)
    (BiologicalProcess): Embedding(446, 128)
    (Activity): Embedding(546, 128)
    (Pathology): Embedding(126, 128)
    (MicroRna): Embedding(46, 128)
    (Protein): Embedding(1301, 128)
    (Rna): Embedding(111, 128)
    (Translocation): Embedding(64, 128)
    (Reaction): Embedding(13, 128)
    (Degradation): Embedding(56, 128)
    (CellSecretion): Embedding(30, 128)
    (CellSurfaceExpression): Embedding(3, 128)
    (Complex): Embedding(373, 128)
    (Composite): Embedding(72, 128)
    (Patient): Embedding(744, 128)
  )
  (conv1): HeteroConv(num_relations=996)
  (conv2): HeteroConv(num_relations=996)
  (rule_layer): FuzzyRuleLayer()
  (gate): Linear(in_features=2, out_features=1, bias=True)
  (classifier): Sequential(
    (0): Linear(in_features=256, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=2, bias=True)
  )
  (rel_weights

## Training

In [22]:
def train_step(model, data, edge_index_dict, rule_features, optimizer, lambdas):
    model.train()
    optimizer.zero_grad()
     # 1. forward Pass
    h_dict, class_out, r = model(edge_index_dict, rule_features)
    
    data['Patient'].y = torch.as_tensor(data['Patient'].y).long().squeeze()
    # 2. classification Loss 
    loss_cls = F.nll_loss(class_out[data['Patient'].train_mask], data['Patient'].y[data['Patient'].train_mask])
    
    # 3. link prediction loss
    loss_lp = 0
    for edge_type in edge_index_dict.keys():
        pos_edge_index = edge_index_dict[edge_type]
        
        # Positive Scores
        pos_scores = model.decode(h_dict, pos_edge_index, edge_type)
        
        # Negative Sampling: Randomly shuffle destination nodes
        neg_edge_index = pos_edge_index.clone()
        neg_edge_index[1] = neg_edge_index[1][torch.randperm(neg_edge_index.size(1))]
        neg_scores = model.decode(h_dict, neg_edge_index, edge_type)
        
        # Ranking Loss: Encourage pos_scores > neg_scores
        loss_lp += -torch.log(torch.sigmoid(pos_scores - neg_scores) + 1e-15).mean()
    
    loss_lp = loss_lp / len(data.edge_types) # to prevent link prediction loss becomes too huge

    # 4. regularizer to ensure rule activations to be 'decisive' (near 0 or 1, not 0.5)
    loss_reg = torch.mean(r * (1 - r)) 

    # 5. Combined Total Loss
    total_loss = (lambdas['cls'] * loss_cls + 
                  lambdas['lp'] * loss_lp + 
                  lambdas['reg'] * loss_reg)
    print(total_loss.grad_fn)
    
    total_loss.backward()
    optimizer.step()

    results = {
        'total_loss': total_loss.item(),
        'cls_loss': loss_cls.item(),
        'lp_loss': loss_lp.item(),
        'reg_loss': loss_reg,
        'rules': r
    }
    
    return results

In [17]:
class LambdaScheduler:
    def __init__(self, total_epochs):
        self.total_epochs = total_epochs

    def get_lambdas(self, epoch):
        # Phase 1: Warming up the KG (Epoch 0-20)
        if epoch < 20:
            return {'cls': 0.1, 'lp': 1.0, 'reg': 0.0}
        
        # Phase 2: Shift focus to Classification (Epoch 21-100)
        elif epoch < 100:
            # Linear ramp for classification: from 0.1 to 1.0
            cls_val = 0.1 + (0.9 * (epoch - 20) / 80)
            return {'cls': cls_val, 'lp': 0.1, 'reg': 0.005}
        
        # Phase 3: Sharpen the Symbolic Rules (Epoch 100+)
        else:
            return {'cls': 1.0, 'lp': 0.05, 'reg': 0.02}

# Usage in your training loop:
scheduler = LambdaScheduler(total_epochs=200)

#### training loop

In [23]:
patient_kg_path = "./data/KG/patient_kg.pkl"

with open(patient_kg_path, 'rb') as f:
    G = pickle.load(f) # patient_kg

# prepare HeteroData
data, new_node_mappings = networkx_to_hetero_data(G)

model = HeteroFireGNN(
    data=data,
    hidden_channels=128,
    out_channels=2,
    heads=2,
    dropout_rate=0.5,
    num_rules=2
)
#x_dict = {nt:data[nt].x for nt in data.node_types}
edge_index_dict = {edge_type: data[edge_type].edge_index for edge_type in data.edge_types}

optimizer = torch.optim.Adam(
        model.parameters(),
        lr=0.005,
        weight_decay=5e-4
    )
result_history = []
for epoch in range(2):
    current_lambdas = scheduler.get_lambdas(epoch)
    
    epoch_result = train_step(model=model,
           data=data,
           edge_index_dict=edge_index_dict,
           rule_features=relevance_features,
           optimizer=optimizer,
           lambdas=current_lambdas)
    result_history.append(epoch_result)
    

In [30]:
print('Rule_gate on all Patient_Protein edges')
edge_gates = result_history[1]['rules']
print('Unique gates:', edge_gates)

Rule_gate on all Patient_Protein edges
Unique gates: tensor([[0.6413],
        [0.6413],
        [0.6413],
        ...,
        [0.6422],
        [0.6422],
        [0.6422]], grad_fn=<IndexBackward0>)


## Learnable edge_coefficient to decide message passing

In [28]:
class EdgeFuzzyLayer(nn.Module):
    """
    Edeg-level fuzzy rule and patient embedding induced edge_coefficient.
    Different patients receiving different edge_coeff scaled messages for the same protein. 
    """
    def __init__(self, num_features=2, patient_emb_dim=64):
        super().__init__()

        # Learnable thresholds and sharpness for feature rules (same to FireGNN)
        self.theta = nn.Parameter(torch.zeros(num_features))
        self.alpha = nn.Parameter(torch.ones(num_features))

        # Convert patient embedding to rule weights in [0,1] => the edge_coeff is decided by protein relevance, expression valiue and patient embedding
        self.patient_weight = nn.Sequential(
            nn.Linear(patient_emb_dim,16),
            nn.ReLU(),
            nn.Linear(16, num_features),
            nn.Sigmoid()
        )
    
    def forward(self, edge_features, patient_embeddings):
        """Generate edge specific and learnable coefficients.

        Args:
            edge_features (Tensor): [num_edges, num_features]
            patient_embeddings (Tensor): [num_edges, patient_emb_dim]

        Returns:
            Tuple[]: edge_coeffs, rule_activations, patient_weights
        """
        # Fuzzy rule activation -> [num_edges, num_features]
        rule_activations = torch.sigmoid(
            self.alpha * (edge_features - self.theta)
        )
        # patient-weight -> [num_edges, num_features]
        patient_weights = self.patient_weight(patient_embeddings)

        # combine features and patient weights to get edge specific coefficient -> [num_edges]
        edge_coeffs = (rule_activations * patient_weights).sum(dim= -1)
        edge_coeffs = torch.sigmoid(edge_coeffs)

        return edge_coeffs, rule_activations, patient_weights

In [59]:
class HeteroFuzzyGAT(nn.Module):
    def __init__(self, data, hidden_channels, out_channels, heads, dropout_rate, num_features=2):
        super().__init__()

        self.dropout_rate = dropout_rate
        self.hidden_channels = hidden_channels
        self.patient_protein_edge_type = ('Patient', 'express', 'Protein')

        # Neura part
        self.embeddings = torch.nn.ModuleDict({
            node_type: torch.nn.Embedding(num_nodes, hidden_channels)
            for node_type, num_nodes in {nt: data[nt].num_nodes for nt in data.node_types}.items()
        })
        self.conv1 = (HeteroConv({
            edge_type: GATConv((-1, -1), hidden_channels, heads, add_self_loops=False)
            for edge_type in data.edge_types
        }))
        self.conv2 = (HeteroConv({
            edge_type: GATConv(hidden_channels*heads, hidden_channels, heads=1, add_self_loops=False)
            for edge_type in data.edge_types
        }))
      
        # Link Prediction Head 
        self.rel_weights = nn.ParameterDict({
            "__".join(edge_type): nn.Parameter(torch.ones(hidden_channels))
            for edge_type in data.edge_types
        })
        for param in self.rel_weights.values():
            torch.nn.init.xavier_uniform_(param.unsqueeze(0))
        
        # classifier
        self.classifier = nn.Sequential(
            nn.Linear(hidden_channels * 2, hidden_channels),
            nn.ReLU(),
            nn.Linear(hidden_channels, out_channels)
        )

        # edge fuzzy fules to get edge_coeffs
        self.edge_fuzzy_rules = EdgeFuzzyLayer(
            num_features=num_features,
            patient_emb_dim=hidden_channels
        )
    
    def forward(self, edge_index_dict, edge_features):

        # Neural learning
        x_dict = {node_type: emb.weight for node_type, emb in self.embeddings.items()}
        h_dict = self.conv1(x_dict, edge_index_dict)
        h_dict = {key: F.elu(v) for key, v in h_dict.items()}
        h_dict = {key: F.dropout(v, p = self.dropout_rate, training=self.training)
                                    for key, v in h_dict.items()}
        h_dict = self.conv2(h_dict, edge_index_dict)
        h_dict = {key: F.elu(v) for key, v in h_dict.items()}
        h_patient = h_dict['Patient']
        h_protein = h_dict['Protein']

        # edge coefficients
        pat_idx, prot_idx = edge_index_dict[self.patient_protein_edge_type]
        patient_embeddings = h_patient[pat_idx] #->[num_edges, hidden_channels]
        edge_coeffs, rule_activations, patient_weights = self.edge_fuzzy_rules(
            edge_features, patient_embeddings
        )
        # selective message passing
        protein_messages = h_protein[prot_idx]
        scaled_messages = protein_messages * edge_coeffs.unsqueeze(1)
        aggregate_to_patient = scatter_sum(
            scaled_messages,
            pat_idx,
            dim=0,
            dim_size=h_patient.size(0)
        )

        # classification
        new_hpatient = torch.cat([h_patient, aggregate_to_patient], dim=1).float()
        logits = self.classifier(new_hpatient)
        log_probs = F.log_softmax(logits, dim=1)

        return h_dict, log_probs, edge_coeffs, rule_activations, patient_weights
    
    def decode(self, h_dict, edge_index, edge_type):
        """Link Prediction Decoder: Predicts probability of edges"""
        u_type, rel, v_type = edge_type
        src, dst = edge_index
        
        h_src = h_dict[u_type][src]
        h_dst = h_dict[v_type][dst]
        
        rel_key = "__".join(edge_type)
        rel_w = self.rel_weights[rel_key]
        return (h_src * rel_w * h_dst).sum(dim=-1)
    
    def get_learned_rules(self):
        return {
            "feature threshold": self.edge_fuzzy_rules.theta,
            "feature sharpness": self.edge_fuzzy_rules.alpha
        }
    

In [56]:
def train_epoch(model, data, edge_index_dict, edge_features, optimizer, lambdas):
    model.train()
    optimizer.zero_grad()
     # 1. forward Pass
    h_dict, log_probs, edge_coeffs, rule_activations, patient_weights = model(edge_index_dict, edge_features)
    
    data['Patient'].y = torch.as_tensor(data['Patient'].y).long().squeeze()
    # 2. classification Loss 
    loss_cls = F.nll_loss(log_probs[data['Patient'].train_mask], data['Patient'].y[data['Patient'].train_mask])
    
    # 3. link prediction loss
    loss_lp = 0
    for edge_type in edge_index_dict.keys():
        pos_edge_index = edge_index_dict[edge_type]
        
        # Positive Scores
        pos_scores = model.decode(h_dict, pos_edge_index, edge_type)
        
        # Negative Sampling: Randomly shuffle destination nodes
        neg_edge_index = pos_edge_index.clone()
        neg_edge_index[1] = neg_edge_index[1][torch.randperm(neg_edge_index.size(1))]
        neg_scores = model.decode(h_dict, neg_edge_index, edge_type)
        
        # Ranking Loss: Encourage pos_scores > neg_scores
        loss_lp += -torch.log(torch.sigmoid(pos_scores - neg_scores) + 1e-15).mean()
    
    loss_lp = loss_lp / len(data.edge_types) # to prevent link prediction loss becomes too huge

    # 4. regularizer to ensure rule activations to be 'decisive' (near 0 or 1, not 0.5)
    loss_reg = torch.mean(edge_coeffs * (1 - edge_coeffs)) 

    # 5. Combined Total Loss
    total_loss = (lambdas['cls'] * loss_cls + 
                  lambdas['lp'] * loss_lp + 
                  lambdas['reg'] * loss_reg)
    print(total_loss.grad_fn)
    
    total_loss.backward()
    optimizer.step()

    results = {
        'total_loss': total_loss.item(),
        'cls_loss': loss_cls.item(),
        'lp_loss': loss_lp.item(),
        'reg_loss': loss_reg,
        'edge_coeff': edge_coeffs
    }
    
    return results

In [ ]:
patient_kg_path = "./data/KG/patient_kg.pkl"

with open(patient_kg_path, 'rb') as f:
    G = pickle.load(f) # patient_kg

# prepare HeteroData
data, new_node_mappings = networkx_to_hetero_data(G)

In [60]:
model = HeteroFuzzyGAT(
    data=data,
    hidden_channels=128,
    out_channels=2,
    heads=2,
    dropout_rate=0.5,
    num_features=2
)
#x_dict = {nt:data[nt].x for nt in data.node_types}
edge_index_dict = {edge_type: data[edge_type].edge_index for edge_type in data.edge_types}

optimizer = torch.optim.Adam(
        model.parameters(),
        lr=0.005,
        weight_decay=5e-4
    )

In [68]:
data[('Patient', 'express', 'Protein')].edge_index.size()

torch.Size([2, 546840])

In [49]:
def get_edge_features(data):
    # prepare edge_features
    # protein_relevance
    patient_idx, protein_idx = data[('Patient', 'express', 'Protein')].edge_index
    protien_relevances = data['Protein'].relevance[protein_idx]
    edge_weights = data[('Patient', 'express', 'Protein')].edge_weight

    edge_features = torch.stack([protien_relevances, edge_weights], dim=1)
    return edge_features

edge_features = get_edge_features(data)
edge_features

tensor([[0.2477, 0.4272],
        [0.2517, 0.2870],
        [0.2229, 0.3142],
        ...,
        [0.2437, 0.3054],
        [0.2873, 0.7307],
        [0.1933, 0.7826]], dtype=torch.float64)

In [61]:
result_history = []
for epoch in range(2):
    current_lambdas = scheduler.get_lambdas(epoch)
    
    epoch_result = train_epoch(model=model,
           data=data,
           edge_index_dict=edge_index_dict,
           edge_features=edge_features,
           optimizer=optimizer,
           lambdas=current_lambdas)
    result_history.append(epoch_result)

In [70]:
data[('Patient', 'express', 'Protein')].edge_index.size()[1]

546840

In [ ]:
class PerEdgeFuzzyLayer(nn.Module):

    def __init__(self, num_edges, num_features=2):
        super().__init__()

        # initialize edge_coeff as 0.5
        self.edge_base_coeffs = nn.Parameter(torch.ones(num_edges)*5)

        # fuzzy rule thresholds and sharpness
        self. theta = nn.Parameter(torch.zeros(num_features))
        self.alpha = nn.Parameter(torch.ones(num_features))
    
    def forward(self, edge_indices, edge_features):
        base_coeffs = self.edge_base_coeffs[edge_indices]
        rule_activations = torch.sigmoid(
            self.alpha * (edge_features - self.theta)
        ).sum(dim=-1)
        edge_coeffs = base_coeffs * rule_activations
        edge_coeffs = torch.sigmoid(edge_coeffs)
        
        return edge_coeffs

class HeteroFuzzyGAT_PerEdge(nn.Module):
    def __init__(self, data, hidden_channels, out_channels, heads, dropout_rate, num_features=2):
        super().__init__()

        self.dropout_rate = dropout_rate
        self.hidden_channels = hidden_channels
        self.patient_protein_edge_type = ('Patient', 'express', 'Protein')

        # Neura part
        self.embeddings = torch.nn.ModuleDict({
            node_type: torch.nn.Embedding(num_nodes, hidden_channels)
            for node_type, num_nodes in {nt: data[nt].num_nodes for nt in data.node_types}.items()
        })
        self.conv1 = (HeteroConv({
            edge_type: GATConv((-1, -1), hidden_channels, heads, add_self_loops=False)
            for edge_type in data.edge_types
        }))
        self.conv2 = (HeteroConv({
            edge_type: GATConv(hidden_channels*heads, hidden_channels, heads=1, add_self_loops=False)
            for edge_type in data.edge_types
        }))
      
        # Link Prediction Head 
        self.rel_weights = nn.ParameterDict({
            "__".join(edge_type): nn.Parameter(torch.ones(hidden_channels))
            for edge_type in data.edge_types
        })
        for param in self.rel_weights.values():
            torch.nn.init.xavier_uniform_(param.unsqueeze(0))
        
        # classifier
        self.classifier = nn.Sequential(
            nn.Linear(hidden_channels * 2, hidden_channels),
            nn.ReLU(),
            nn.Linear(hidden_channels, out_channels)
        )

        # edge fuzzy fules to get edge_coeffs
        self.edge_fuzzy_rules = PerEdgeFuzzyLayer(
            num_edges=data[('Patient', 'express', 'Protein')].edge_index.size()[1],
            num_features=num_features
        )
    
    def forward(self, edge_index_dict, edge_features):

        # Neural learning
        x_dict = {node_type: emb.weight for node_type, emb in self.embeddings.items()}
        h_dict = self.conv1(x_dict, edge_index_dict)
        h_dict = {key: F.elu(v) for key, v in h_dict.items()}
        h_dict = {key: F.dropout(v, p = self.dropout_rate, training=self.training)
                                    for key, v in h_dict.items()}
        h_dict = self.conv2(h_dict, edge_index_dict)
        h_dict = {key: F.elu(v) for key, v in h_dict.items()}
        h_patient = h_dict['Patient']
        h_protein = h_dict['Protein']

        # edge coefficients
        pat_idx, prot_idx = edge_index_dict[self.patient_protein_edge_type]
        
        edge_indices = torch.arange(len(pat_idx))
        edge_coeffs= self.edge_fuzzy_rules(edge_indices, edge_features)
        # selective message passing
        protein_messages = h_protein[prot_idx]
        scaled_messages = protein_messages * edge_coeffs.unsqueeze(-1)
        aggregate_to_patient = scatter_sum(
            scaled_messages,
            pat_idx,
            dim=0,
            dim_size=h_patient.size(0)
        )

        # classification
        new_hpatient = torch.cat([h_patient, aggregate_to_patient], dim=1).float()
        logits = self.classifier(new_hpatient)
        log_probs = F.log_softmax(logits, dim=1)

        return h_dict, log_probs, edge_coeffs
    
    def decode(self, h_dict, edge_index, edge_type):
        """Link Prediction Decoder: Predicts probability of edges"""
        u_type, rel, v_type = edge_type
        src, dst = edge_index
        
        h_src = h_dict[u_type][src]
        h_dst = h_dict[v_type][dst]
        
        rel_key = "__".join(edge_type)
        rel_w = self.rel_weights[rel_key]
        return (h_src * rel_w * h_dst).sum(dim=-1)
    
    def get_learned_rules(self):
        return {
            "feature threshold": self.edge_fuzzy_rules.theta,
            "feature sharpness": self.edge_fuzzy_rules.alpha
        }
    def get_edge_coeff_statistics(self):
        with torch.no_grad():
            base_coeffs = torch.sigmoid(self.edge_fuzzy_rules.edge_base_coeffs)
            return {
                'mean': base_coeffs.mean().item(),
                'std': base_coeffs.std().item(),
                'min': base_coeffs.min().item(),
                'max': base_coeffs.max().item(),
                'num_high_coeffs': (base_coeffs > 0.7).sum().item(),  # "Strong" edges
                'num_low_coeffs': (base_coeffs < 0.3).sum().item(),
            }
    